# Step 7c · second adversarial round for DeBERTa (30% budget)
Round 1 trained on 15%-budget rewrites; a 30%-budget re-attack still found 9 genuine evasions (7 on held-out seeds).
Round 2 adds the 30%-budget rewrites of **train seeds** (from the attacks on BASE and on the round-1 ADV model) to the
adversarial training data, then re-attacks at 30% / 100 steps and runs the agent check.
Held-out test seeds (r1t, r2t) are never trained on, so their re-attack is a clean test.
~1.5–2 h. **Settings:** GPU **T4 x2** · Internet **On** · Secrets **Hugging Face** + **mcp-git** · Save & Run All.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SOURCES = "[results/wb1/whitebox.jsonl,results/wb1e5/whitebox.jsonl,results/wb3_base_b30_deb/whitebox.jsonl,results/wb3_adv_b30_deb/whitebox.jsonl]"
ATTACK = "whitebox.max_frac=0.30 whitebox.steps=100"
GPU_DEPS = True
MIN_COMMIT_FILE = "results/wb3_adv_b30_deb/whitebox.jsonl"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push the latest commit first (needs results/wb3_*)"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2"
!python scripts/adv_train.py --tag adv2_deb --set 'advtrain.sources={SOURCES}'


In [ ]:
import glob, os
adv_run = sorted(glob.glob("/kaggle/working/outputs/adv_train/*"))[-1]
ckpt = os.path.join(adv_run, "ckpt_adv")
assert os.path.exists(os.path.join(ckpt, "detector.json")), ckpt
print(open(os.path.join(adv_run, "adv_results.md")).read())
!python scripts/attack_whitebox.py --tag wb4_adv2_b30_deb --set whitebox.detector_ckpt={ckpt} {ATTACK}


In [ ]:
import glob, json
run = sorted(glob.glob("/kaggle/working/outputs/attack_whitebox/*"))[-1]
s = json.load(open(f"{run}/summary.json"))
print(os.path.basename(run), "| evaded", s["n_evaded"], "| genuine", s.get("n_genuine_evasions"), "| of", s["n_items"])
print(json.dumps(s["by_round"]))
